# 02c — EDA: clustering jerárquico y PCA exploratorio (edad × sexo)

**Sub-fase 2c — CRISP-DM: Data Understanding (extensión no supervisada)** ·
Rama `feature/fase-2c-eda-clustering`, abierta a pedido del usuario tras cerrar
el Loop C (`02_eda.ipynb`).

**Objetivo.** Explorar si un clustering jerárquico sobre el perfil bioquímico,
estratificado por banda de edad y sexo, revela (a) patrones sugerentes de
heterogeneidad etiológica/severidad, y (b) puntos de corte candidatos para
`Sgpt` (ALT) comparables contra los umbrales de literatura ya adoptados
(`docs/adr/0004-umbrales-referencia-sexo-especificos.md`).

**⚠️ Alcance y límites, acordados con el usuario antes de escribir código
(ver `AGENTS.md`, checkpoint 2026-07-28):**

1. Esto es **exploración de hipótesis, no un modelo**. No se entrena ningún
   clasificador ni se calculan métricas de clasificación — sigue vigente el
   límite del PRD (§2.3) para las Fases 0–3.
2. Cualquier "punto de corte candidato" que salga de acá es **señal
   exploratoria derivada de este dataset**, explícitamente **no validada
   clínicamente**. Nunca reemplaza los umbrales de Prati/ACG/AASLD del ADR
   0004 — en el peor caso los contradice, y ahí se explica por qué en vez de
   descartar la contradicción.
3. Notebook separado de `02_eda.ipynb` a propósito: mezclar clustering
   exploratorio con la EDA que responde a la rúbrica (T1–T5) diluiría cuál
   celda responde a qué.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram
from sklearn.preprocessing import StandardScaler

from src.config import (
    NUMERIC_COLS,
    SKEWED_COLS,
    LIVER_AXES,
    ALT_ULN_BY_SEX,
    MIN_CLUSTER_SIZE,
    MIN_STRATUM_SIZE_FOR_CLUSTERING,
    INK_PRIMARY,
    INK_SECONDARY,
    GRID_COLOR,
    SURFACE,
    SEX_COLORS,
    DPI,
)
from src.utils import (
    load_raw_data,
    save_figure,
    separation_by_variable,
    flag_biochemical_violations,
    assign_age_sex_stratum,
    hierarchical_cluster_cut,
    cluster_sizes,
    de_ritis_ratio,
)

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": INK_SECONDARY,
    "axes.labelcolor": INK_PRIMARY,
    "text.color": INK_PRIMARY,
    "xtick.color": INK_SECONDARY,
    "ytick.color": INK_SECONDARY,
    "grid.color": GRID_COLOR,
})

## Preparación — exclusiones de calidad conocidas y estratificación

**Exclusiones.** Este notebook trabaja sobre datos **crudos, sin imputar**
(la imputación formal es tarea de la Fase 3, todavía sin código — ver
`feature/fase-3-preprocessing`). Pero incluir filas con violaciones
bioquímicas ya documentadas inyectaría ruido conocido en un análisis que
busca justamente patrones de daño real:

- **4 filas con `A/G Ratio` faltante** (Q2) — `StandardScaler` no acepta
  `NaN`.
- **3 filas con `DB > TB`** (Q8, Loop C) — violación de plausibilidad ya
  documentada; su `TB`/`DB` no son confiables para ningún análisis.

Se excluyen ambas (7 filas, n efectivo 576) **solo para este notebook
exploratorio** — la Fase 3 decide su tratamiento definitivo (imputación) de
forma independiente, ver `AGENTS.md`.

In [2]:
df = load_raw_data()

missing_ag = df["A/G Ratio"].isna()
db_gt_tb = flag_biochemical_violations(df)["db_gt_tb"]
excluded = missing_ag | db_gt_tb

print(f"Filas excluidas por calidad: {excluded.sum()} "
      f"(A/G faltante: {missing_ag.sum()}, DB>TB: {db_gt_tb.sum()})")
print(f"n disponible: {(~excluded).sum()}")

work = df[~excluded].copy()
work["Stratum"] = assign_age_sex_stratum(work)

stratum_counts = work["Stratum"].value_counts()
STRATA = sorted(stratum_counts[stratum_counts >= MIN_STRATUM_SIZE_FOR_CLUSTERING].index)
STRATA_EXCLUDED = sorted(stratum_counts[stratum_counts < MIN_STRATUM_SIZE_FOR_CLUSTERING].index)

print(f"\n--- Estratos (umbral de clustering: n >= {MIN_STRATUM_SIZE_FOR_CLUSTERING}) ---")
for s in stratum_counts.index:
    marca = "ANALIZADO" if s in STRATA else "EXCLUIDO "
    print(f"  [{marca}] {s:<22} n={stratum_counts[s]}")

print(f"\nEstratos que entran al clustering: {len(STRATA)} "
      f"| n total analizado: {int(stratum_counts[STRATA].sum())}")

Filas excluidas por calidad: 7 (A/G faltante: 4, DB>TB: 3)
n disponible: 576

--- Estratos (umbral de clustering: n >= 30) ---
  [ANALIZADO] 40-59 · Male           n=156
  [ANALIZADO] 18-39 · Male           n=150
  [ANALIZADO] 60-120 · Male          n=113
  [ANALIZADO] 40-59 · Female         n=65
  [ANALIZADO] 18-39 · Female         n=47
  [EXCLUIDO ] 0-17 · ambos sexos     n=25
  [EXCLUIDO ] 60-120 · Female        n=20

Estratos que entran al clustering: 5 | n total analizado: 531


**Interpretación — 7 estratos, pero solo 5 admiten clustering.**

La estratificación produce 7 grupos: las 3 bandas adultas divididas por sexo, más la banda pediátrica (0–17) sin dividir, porque ninguno de sus dos sexos alcanza `MIN_STRATUM_SIZE_FOR_SEX_SPLIT`.

**Dos de esos 7 quedan fuera del clustering** (`MIN_STRATUM_SIZE_FOR_CLUSTERING = 30`):

| Estrato excluido | n | Por qué |
|---|---|---|
| `0-17 · ambos sexos` | 25 | Muestra insuficiente |
| `60-120 · Female` | 20 | Muestra insuficiente |

**Por qué se excluyen en vez de reportarlos con un caveat.** Con muestras de ese tamaño el dendrograma no encuentra grupos: aísla individuos. En la primera versión de este notebook, esos dos estratos producían "clusters" de **1 y 2 personas**, y sobre ellos se calculaban medianas — la mediana de una sola observación **es** esa observación. Reportarla como "mediana del cluster" da una falsa apariencia de agregación estadística. Un caveat al pie no arregla eso; lo correcto es no calcularlo.

> ⚠️ **Esto tiene una consecuencia incómoda que hay que decir en voz alta:** de los dos estratos excluidos, uno es el de **mujeres de 60+**. Este análisis, en un proyecto cuyo eje es la equidad por sexo, **no puede decir nada sobre las mujeres mayores** — porque solo hay 20 en todo el dataset.
>
> No es un defecto de este notebook: es una consecuencia directa del desbalance de sexo (Q5: 441 hombres vs 142 mujeres). **El desbalance no solo sesga a un modelo futuro — también limita qué preguntas se pueden responder sobre las mujeres, incluso en un análisis puramente descriptivo.** Queda registrado como limitación para la Fase 5.

Los 5 estratos analizados suman **531 personas**.

## Experimento 1 — clustering sobre las 9 variables, con análisis de sensibilidad

Las 9 variables numéricas (`NUMERIC_COLS`), estandarizadas **dentro de cada
estrato** (para comparar a cada persona contra su propio grupo de referencia
edad-sexo, no contra la población general), clustering jerárquico de Ward, y
corte por el mayor salto de distancia de fusión — sin fijar `k` a mano.

### Por qué este experimento se corre dos veces

`StandardScaler` centra y escala, pero **no cambia la forma** de la
distribución. Y T4 (`02_eda.ipynb`) documentó que estas variables tienen una
asimetría extrema:

| Variable | *Skewness* | El paciente más alto queda a… |
|---|---|---|
| `Sgot` | **10.5** | ~17 desviaciones estándar |
| `Sgpt` | 6.5 | ~10 desviaciones |
| `TB` | 4.9 | ~11 desviaciones |

El clustering de Ward mide **distancias euclidianas**. Una persona a 17
desviaciones está tan lejos de todas las demás que el mayor salto del
dendrograma es el de **unirla al resto** — y el corte automático la aísla,
devolviendo un "cluster" de una sola persona. Eso no es un grupo: es un
*outlier* con otro nombre.

Por eso se corren **dos variantes** y se reportan ambas:

- **(a) Crudo** — las 9 variables tal cual, estandarizadas.
- **(b) `log1p`** — con logaritmo aplicado a las 5 variables sesgadas
  (`SKEWED_COLS`) antes de estandarizar. Comprime la cola derecha sin borrar
  el orden entre pacientes. Es la transformación estándar para variables de
  laboratorio antes de calcular distancias, y es la que el Experimento 2 ya
  usaba.

Además, ambas variantes exigen ahora un **tamaño mínimo de cluster**
(`MIN_CLUSTER_SIZE = 5`): `hierarchical_cluster_cut` recorre los saltos de
mayor a menor y toma el primero cuyo cluster más chico llegue a ese tamaño.

> **La comparación entre (a) y (b) no es un control de calidad interno — es
> parte del resultado.** Si el hallazgo cambia según la transformación, eso
> hay que reportarlo, no esconderlo eligiendo la variante que más guste.

In [3]:
def run_experiment_1(stratum: str, use_log: bool):
    """Clustering de un estrato sobre las 9 variables, con o sin log1p."""
    sub = work.loc[work["Stratum"] == stratum].copy()
    M = sub[NUMERIC_COLS].copy()
    if use_log:
        M[SKEWED_COLS] = np.log1p(M[SKEWED_COLS])
    X = StandardScaler().fit_transform(M)
    Z, labels, cut, k = hierarchical_cluster_cut(X, min_cluster_size=MIN_CLUSTER_SIZE)
    sub["Cluster"] = labels
    return sub, Z, k, cluster_sizes(labels)


exp1_results = {}
exp1_rows = []

for stratum in STRATA:
    for use_log in (False, True):
        variante = "log1p" if use_log else "crudo"
        sub, Z, k, sizes = run_experiment_1(stratum, use_log)
        exp1_results[(stratum, variante)] = {"sub": sub, "Z": Z, "k": k, "sizes": sizes}

        # cluster "severo" = el de mayor mediana de TB (eje de excrecion biliar)
        severe_id = sub.groupby("Cluster")["TB"].median().idxmax()
        severe = sub[sub["Cluster"] == severe_id]
        rest = sub[sub["Cluster"] != severe_id]

        exp1_rows.append({
            "Estrato": stratum,
            "variante": variante,
            "n": len(sub),
            "k": k,
            "tamanos": str(sizes),
            "min_size_ok": min(sizes) >= MIN_CLUSTER_SIZE,
            "n_severo": len(severe),
            "TB_med_severo": round(severe["TB"].median(), 1),
            "TB_med_resto": round(rest["TB"].median(), 1),
            "pct_dx_severo": round(100 * (severe["Selector"] == 1).mean(), 1),
            "pct_dx_resto": round(100 * (rest["Selector"] == 1).mean(), 1),
        })

exp1_summary = pd.DataFrame(exp1_rows).set_index(["Estrato", "variante"])
exp1_summary

n  k            tamanos  min_size_ok  n_severo  \
Estrato        variante                                                     
18-39 · Female crudo      47  2            [41, 6]         True         6   
               log1p      47  2           [37, 10]         True        10   
18-39 · Male   crudo     150  2          [131, 19]         True        19   
               log1p     150  2          [121, 29]         True        29   
40-59 · Female crudo      65  5  [31, 19, 6, 5, 4]        False         4   
               log1p      65  2           [42, 23]         True        23   
40-59 · Male   crudo     156  2          [136, 20]         True        20   
               log1p     156  2           [78, 78]         True        78   
60-120 · Male  crudo     113  2          [100, 13]         True        13   
               log1p     113  3       [58, 33, 22]         True        22   

                         TB_med_severo  TB_med_resto  pct_dx_severo  \
Estrato        variante                                               
18-39 · Female crudo               1.1           0.8           83.3   
               log1p               2.2           0.8           60.0   
18-39 · Male   crudo              15.9           1.0          100.0   
               log1p               8.7           0.9           93.1   
40-59 · Female crudo              23.2           0.9          100.0   
               log1p               3.0           0.8          100.0   
40-59 · Male   crudo              17.0           1.0          100.0   
               log1p               3.2           0.8           93.6   
60-120 · Male  crudo              11.5           1.3          100.0   
               log1p               8.2           1.1          100.0   

                         pct_dx_resto  
Estrato        variante                
18-39 · Female crudo             48.8  
               log1p             51.4  
18-39 · Male   crudo             64.9  
               log1p             63.6  
40-59 · Female crudo             73.8  
               log1p             61.9  
40-59 · Male   crudo             75.7  
               log1p             64.1  
60-120 · Male  crudo             74.0  
               log1p             71.4

In [4]:
fig, axes = plt.subplots(len(STRATA), 2, figsize=(12, 3.1 * len(STRATA)))

for fila, stratum in enumerate(STRATA):
    for col, variante in enumerate(["crudo", "log1p"]):
        ax = axes[fila, col]
        r = exp1_results[(stratum, variante)]
        dendrogram(r["Z"], ax=ax, no_labels=True, color_threshold=0,
                   above_threshold_color=INK_SECONDARY)
        ax.set_title(f"{stratum} — {variante}  (k={r['k']}, tamaños={r['sizes']})",
                     fontsize=9)
        ax.set_ylabel("distancia (Ward)", fontsize=8)
        ax.tick_params(labelsize=7)

fig.suptitle("Experimento 1 — sensibilidad del clustering a la transformación\n"
             "Izquierda: variables crudas · Derecha: log1p sobre las 5 sesgadas",
             y=1.0, fontsize=11.5, fontweight="bold")
fig.tight_layout()
save_figure(fig, "fase2c_exp1_dendrogramas.png")
plt.show()

**Interpretación — el clustering encuentra estructura, pero *qué* estructura depende de la transformación.**

### Lo que cada variante encuentra

| Estrato | Variante | Tamaños | n severo | `TB` severo | `TB` resto | % dx severo | % dx resto |
|---|---|---|---|---|---|---|---|
| 18-39 · Female | crudo | `[41, 6]` | 6 | 1.1 | 0.8 | 83.3 | 48.8 |
| 18-39 · Female | **log1p** | `[37, 10]` | 10 | 2.2 | 0.8 | 60.0 | 51.4 |
| 18-39 · Male | crudo | `[131, 19]` | 19 | **15.9** | 1.0 | 100.0 | 64.9 |
| 18-39 · Male | **log1p** | `[121, 29]` | 29 | 8.7 | 0.9 | 93.1 | 63.6 |
| 40-59 · Female | crudo | `[31, 19, 6, 5, 4]` | 4 | **23.2** | 0.9 | 100.0 | 73.8 |
| 40-59 · Female | **log1p** | `[42, 23]` | 23 | 3.0 | 0.8 | 100.0 | 61.9 |
| 40-59 · Male | crudo | `[136, 20]` | 20 | **17.0** | 1.0 | 100.0 | 75.7 |
| 40-59 · Male | **log1p** | `[78, 78]` | 78 | 3.2 | 0.8 | 93.6 | 64.1 |
| 60-120 · Male | crudo | `[100, 13]` | 13 | 11.5 | 1.3 | 100.0 | 74.0 |
| 60-120 · Male | **log1p** | `[58, 33, 22]` | 22 | 8.2 | 1.1 | 100.0 | 71.4 |

### Las dos variantes responden preguntas distintas

**Con variables crudas**, el clustering aísla un grupo **pequeño y extremo**: 4 a 20 personas con `TB` entre 11.5 y 23.2 mg/dL — es decir, **10 a 19 veces el límite superior normal** (1.2 mg/dL) — y prácticamente el 100% diagnosticado. Son los pacientes con ictericia franca y descompensación evidente.

**Con `log1p`**, el clustering encuentra un grupo **más amplio y moderado**: 10 a 78 personas con `TB` entre 2.2 y 8.7, y porcentajes de diagnóstico de 60% a 100%. Ya no son solo los casos extremos, sino la franja de alteración moderada.

Ninguna de las dos es "la correcta". **La cruda responde "¿quiénes están claramente descompensados?"; la logarítmica responde "¿dónde está la separación principal de la población?"** Son preguntas distintas y ambas legítimas.

### El caso que más cambia: `40-59 · Male`

Pasa de `[136, 20]` —un grupo chico de severidad— a **`[78, 78]`**, dos mitades exactamente parejas. **Ese no es el mismo hallazgo.** Sin transformar, la lectura sería "existe un subgrupo severo minoritario"; con `log1p`, "la población se divide en dos mitades por nivel de alteración".

Reportar solo una de las dos habría sido presentar como propiedad de los datos algo que es, en buena parte, **consecuencia de una decisión de preprocesamiento**.

### El diagnóstico más duro: `40-59 · Female` con variables crudas

La columna `min_size_ok` marca **`False`** en esa fila. Significa que `hierarchical_cluster_cut` recorrió **todos** los cortes posibles del dendrograma y **ninguno** produce clusters donde el más chico llegue a 5 personas — tuvo que caer al corte por defecto, devolviendo `[31, 19, 6, 5, 4]`.

Es la evidencia más directa de que la estructura cruda está dominada por valores extremos: **no existe ninguna forma de cortar ese dendrograma que dé grupos, en vez de individuos sueltos.** Con `log1p`, el mismo estrato produce `[42, 23]` sin problema.

### Qué queda establecido, pese a la sensibilidad

A pesar de que los tamaños y las medianas cambian mucho entre variantes, **una cosa se mantiene en las dos y en los cinco estratos**: el cluster con `TB` más alta tiene **siempre** un porcentaje de diagnóstico positivo mayor que el resto — entre 60% y 100% frente a 49%–76%.

Es decir: **el clustering no supervisado, que nunca ve la etiqueta `Selector`, reagrupa personas de forma que se correlaciona con el diagnóstico del médico.** Eso sí es un hallazgo robusto a la transformación, y es el que vale la pena reportar.

### Por qué esto no da un punto de corte

El clustering separa por el **perfil combinado de 9 variables**, no por un eje único. Al proyectar esos clusters sobre `Sgpt` (ALT) solamente, los rangos **se superponen**. Un punto de corte univariado derivado de acá sería inventado, no medido. El Experimento 2 pone a prueba exactamente eso.

## Experimento 2 — clustering restringido al eje de daño celular (`Sgpt`, `Sgot`)

Pedido explícito del usuario: repetir el clustering usando solo las 2
variables del eje de daño celular (`LIVER_AXES["dano_celular"]`), para ver
si reducir la dimensionalidad resuelve un corte más fino entre "sano" y
"levemente alterado" — justo el rango donde operan los umbrales de ALT de la
literatura.

Se prueban **dos variantes**: sobre los valores crudos, y sobre
`log1p(valor)` antes de estandarizar. La segunda variante no es arbitraria:
T4 (`02_eda.ipynb`) ya documentó que `Sgpt`/`Sgot` tienen una distribución
muy asimétrica con cola larga — el logaritmo es la transformación estándar
para ese tipo de variable de laboratorio antes de calcular distancias.

In [5]:
damage_cols = LIVER_AXES["dano_celular"]
exp2_results = {}
exp2_rows = []

for stratum in STRATA:
    sub = work.loc[work["Stratum"] == stratum].copy()

    X_log = StandardScaler().fit_transform(np.log1p(sub[damage_cols]))
    Z, labels, cut, k = hierarchical_cluster_cut(X_log, min_cluster_size=MIN_CLUSTER_SIZE)
    sub["ClusterLog"] = labels
    exp2_results[stratum] = {"sub": sub, "Z": Z, "k": k, "sizes": cluster_sizes(labels)}

    # cluster de menor mediana de Sgpt = candidato a "linea de base" del estrato
    base_id = sub.groupby("ClusterLog")["Sgpt"].median().idxmin()
    base_vals = sub.loc[sub["ClusterLog"] == base_id, "Sgpt"]
    rest_vals = sub.loc[sub["ClusterLog"] != base_id, "Sgpt"]

    overlap = bool(base_vals.max() >= rest_vals.min()) if len(rest_vals) else np.nan
    candidate = np.nan if overlap else (base_vals.max() + rest_vals.min()) / 2

    sexo = sub["Gender"].iloc[0] if sub["Gender"].nunique() == 1 else None
    uln = ALT_ULN_BY_SEX.get(sexo, np.nan)

    exp2_rows.append({
        "Estrato": stratum,
        "n": len(sub),
        "k": k,
        "tamanos": str(cluster_sizes(labels)),
        "n_base": len(base_vals),
        "Sgpt_max_base": base_vals.max(),
        "Sgpt_min_resto": rest_vals.min() if len(rest_vals) else np.nan,
        "rangos_se_superponen": overlap,
        "corte_candidato": round(candidate, 1) if not np.isnan(candidate) else np.nan,
        "ULN_literatura": uln,
        "veces_el_ULN": round(candidate / uln, 1) if not np.isnan(candidate) else np.nan,
    })

exp2_summary = pd.DataFrame(exp2_rows).set_index("Estrato")
exp2_summary

,n,k,tamanos,n_base,Sgpt_max_base,Sgpt_min_resto,rangos_se_superponen,corte_candidato,ULN_literatura,veces_el_ULN
Estrato,,,,,,,,,,
18-39 · Female,47,2,"[40, 7]",40,90,70,True,NaN,19,NaN
18-39 · Male,150,3,"[75, 63, 12]",75,78,24,True,NaN,30,NaN
40-59 · Female,65,2,"[57, 8]",57,96,110,False,103.0,19,5.4
40-59 · Male,156,2,"[118, 38]",118,88,39,True,NaN,30,NaN
60-120 · Male,113,2,"[86, 27]",86,116,42,True,NaN,30,NaN


In [6]:
n_filas = (len(STRATA) + 1) // 2
fig, axes = plt.subplots(n_filas, 2, figsize=(12, 3.4 * n_filas))
axes = axes.ravel()

for ax, stratum in zip(axes, STRATA):
    r = exp2_results[stratum]
    dendrogram(r["Z"], ax=ax, no_labels=True, color_threshold=0,
               above_threshold_color=INK_SECONDARY)
    ax.set_title(f"{stratum}  (n={len(r['sub'])}, k={r['k']}, tamaños={r['sizes']})",
                 fontsize=9.5)
    ax.set_ylabel("distancia (Ward, log1p)", fontsize=8)
    ax.tick_params(labelsize=7)

for ax in axes[len(STRATA):]:
    ax.axis("off")

fig.suptitle("Experimento 2 — dendrogramas sobre Sgpt + Sgot (log1p), por estrato edad-sexo",
             y=1.0, fontsize=11.5, fontweight="bold")
fig.tight_layout()
save_figure(fig, "fase2c_exp2_dendrogramas.png")
plt.show()

**Interpretación — reducir a 2 variables no resuelve el corte, y la razón es más interesante que "probar otra transformación".**

La columna `rangos_se_superponen` muestra que en **4 de los 5 estratos** los rangos de `Sgpt` de los clusters **siguen superpuestos**, incluso restringiendo el análisis a las dos variables del eje de daño celular y aplicando `log1p`. El único estrato sin superposición (`40-59 · Female`) produce un corte candidato de **103 U/L**, que es **5.4 veces** el ULN femenino de la literatura (19 U/L).

O sea: donde el clustering sí separa limpiamente, separa en un punto **muy por encima** de donde las guías clínicas ponen el límite de normalidad.

### La razón no es un problema de método — es la composición de la muestra

El ILPD es una cohorte de **hospital**: el 71% de los 583 pacientes ya tiene diagnóstico positivo. No es una muestra poblacional con una franja amplia de personas verdaderamente sanas, que es exactamente como Prati et al. (2002) y las guías ACG/AASLD calibraron *sus* umbrales.

Un clustering no supervisado busca **la separación más fuerte que exista en los datos**. En una cohorte mayoritariamente enferma, esa separación más fuerte es **"leve vs. severo"**, no **"sano vs. enfermo"**. El grupo verdaderamente sano —el que el umbral de literatura necesita como referencia— está sub-representado aquí para que el clustering pueda aislarlo de forma confiable.

Dicho de otro modo: **le estamos pidiendo al algoritmo que encuentre el borde de un país cuando casi todos los puntos del mapa están tierra adentro.**

### Esto refuerza, no contradice, la decisión del ADR 0004

El ADR 0004 ya había decidido adoptar umbrales de estudios poblacionales en vez de derivarlos de este dataset. Ese razonamiento era **teórico**: los umbrales requieren una población de referencia sana, y esta no lo es.

Ahora hay **evidencia empírica directa** de que esa decisión era correcta: se intentó derivar cortes de estos datos, con dos configuraciones distintas de variables y con la transformación adecuada, y **no se puede**. Un resultado negativo bien documentado vale tanto como uno positivo — y este cierra la pregunta en vez de dejarla abierta.

### Consistencia con el Experimento 1

Nótese que este experimento **siempre** usó `log1p`, y su justificación (la asimetría documentada en T4) es la misma que ahora se aplica al Experimento 1. En la primera versión de este notebook los dos experimentos usaban criterios distintos sin explicar por qué — esa inconsistencia quedó corregida.

## Control de sesgo del propio análisis — ¿es `Sgpt` (ALT) realmente la variable más importante?

El Experimento 2 se restringió a `Sgpt`/`Sgot`, y tanto el ADR 0004 como el análisis de sensibilidad de umbrales de `02_eda.ipynb` giran también en torno a `Sgpt`. **ALT aparece por todas partes en este proyecto.** Conviene preguntarse si eso refleja los datos o refleja cómo construimos el análisis.

### Por qué ALT domina la narrativa

Hay tres razones, y **solo la primera es clínica**:

1. **ALT es la enzima más específica del hígado** *(razón legítima)*. ALT vive casi exclusivamente en el hepatocito; AST también está en corazón y músculo. Un ALT elevado apunta al hígado con menos ambigüedad.
2. **Es la única variable con umbrales por sexo bien establecidos** *(la razón de verdad)*. Prati et al. (2002) estudiaron ALT específicamente, y ACG 2017 / AASLD 2023 adoptaron cortes diferenciados para ALT. Para el resto, `docs/fuentes/Consulta_1.md` es explícita: *"para ALP, proteína total y bilirrubina directa, la evidencia sur-asiática específica es escasa, inconsistente… y en algunos casos (bilirrubina directa) no existe ningún estudio de referencia validado localmente"*.
3. **Efecto acumulativo**: el Experimento 2 la puso al frente una tercera vez.

La razón 2 es un **sesgo de disponibilidad en el diseño del análisis**: se estudió donde había evidencia publicada, no donde estaba la señal más fuerte. Eso no invalida nada — pero hay que decirlo, en vez de dejar que el lector asuma que ALT es la variable dominante del dataset.

### La verificación

Se compara, variable por variable, qué tan distintas son las distribuciones entre diagnosticados y no diagnosticados, con el **delta de Cliff** (tamaño de efecto basado en rangos, robusto a la asimetría documentada en T4).

> **Alcance:** es estadística descriptiva comparando dos grupos. **No se entrena ningún modelo ni se calcula ninguna métrica de clasificación** — el límite del §2.3 del PRD se mantiene.

In [7]:
print("=== Separacion diagnosticados vs no diagnosticados, por variable ===")
print("(delta de Cliff: 0 = indistinguibles | <0.15 despreciable | 0.15-0.33 pequeno")
print(" 0.33-0.47 mediano | >0.47 grande)\n")

sep_global = separation_by_variable(df, NUMERIC_COLS)
display(sep_global)

print("\n=== Mismo calculo, estratificado por sexo ===")
sep_por_sexo = {}
for sexo in ("Female", "Male"):
    sub = df[df["Gender"] == sexo]
    sep_por_sexo[sexo] = separation_by_variable(sub, NUMERIC_COLS)
    print(f"\n--- {sexo} (n={len(sub)}) ---")
    display(sep_por_sexo[sexo])

print("\n=== Puesto de Sgpt (ALT) en cada ranking ===")
for etiqueta, tabla in [("global", sep_global), *sep_por_sexo.items()]:
    puesto = int(tabla.index[tabla["variable"] == "Sgpt"][0])
    lider = tabla.iloc[0]
    print(f"  {etiqueta:>7}: ALT en el puesto {puesto} de {len(tabla)}  "
          f"| lidera {lider['variable']} (delta={lider['separacion']:.3f})")

=== Separacion diagnosticados vs no diagnosticados, por variable ===
(delta de Cliff: 0 = indistinguibles | <0.15 despreciable | 0.15-0.33 pequeno
 0.33-0.47 mediano | >0.47 grande)



,variable,mediana_dx_positivo,mediana_dx_negativo,cliffs_delta,separacion
puesto,,,,,
1,Sgot,52.50,29.0,0.394,0.394
2,TB,1.40,0.8,0.387,0.387
3,DB,0.50,0.2,0.372,0.372
4,Sgpt,41.00,27.0,0.371,0.371
5,Alkphos,229.00,186.0,0.349,0.349
6,A/G Ratio,0.90,1.0,-0.240,0.240
7,ALB,3.00,3.4,-0.213,0.213
8,Age,46.00,40.0,0.165,0.165
9,TP,6.55,6.6,-0.041,0.041



=== Mismo calculo, estratificado por sexo ===

--- Female (n=142) ---


,variable,mediana_dx_positivo,mediana_dx_negativo,cliffs_delta,separacion
puesto,,,,,
1,TB,0.9,0.80,0.322,0.322
2,Alkphos,203.5,188.00,0.300,0.300
3,DB,0.2,0.20,0.281,0.281
4,Sgot,33.0,27.00,0.245,0.245
5,Sgpt,27.0,24.00,0.242,0.242
6,A/G Ratio,0.9,1.00,-0.183,0.183
7,TP,6.8,6.75,0.076,0.076
8,Age,45.0,39.50,0.066,0.066
9,ALB,3.3,3.25,-0.064,0.064



--- Male (n=441) ---


,variable,mediana_dx_positivo,mediana_dx_negativo,cliffs_delta,separacion
puesto,,,,,
1,Sgot,56.00,30.0,0.425,0.425
2,TB,1.70,0.8,0.402,0.402
3,Sgpt,44.50,28.0,0.399,0.399
4,DB,0.75,0.2,0.391,0.391
5,Alkphos,231.50,185.0,0.350,0.350
6,A/G Ratio,0.90,1.0,-0.261,0.261
7,ALB,3.00,3.5,-0.257,0.257
8,Age,47.00,40.0,0.196,0.196
9,TP,6.40,6.5,-0.078,0.078



=== Puesto de Sgpt (ALT) en cada ranking ===
   global: ALT en el puesto 4 de 9  | lidera Sgot (delta=0.394)
   Female: ALT en el puesto 5 de 9  | lidera TB (delta=0.322)
     Male: ALT en el puesto 3 de 9  | lidera Sgot (delta=0.425)


**Interpretación — ALT no es la variable más separadora, y conviene decirlo.**

### El ranking

| Puesto | Variable | Mediana dx+ | Mediana dx− | Delta de Cliff |
|---|---|---|---|---|
| 1 | `Sgot` (AST) | 52.5 | 29.0 | **0.394** |
| 2 | `TB` (bilirrubina total) | 1.40 | 0.80 | **0.387** |
| 3 | `DB` (bilirrubina directa) | 0.50 | 0.20 | **0.372** |
| **4** | **`Sgpt` (ALT)** | 41.0 | 27.0 | **0.371** |
| 5 | `Alkphos` (ALP) | 229 | 186 | 0.349 |
| 6 | `A/G Ratio` | 0.90 | 1.00 | −0.240 |
| 7 | `ALB` (albúmina) | 3.00 | 3.40 | −0.213 |
| 8 | `Age` | 46 | 40 | 0.165 |
| 9 | `TP` (proteína total) | 6.55 | 6.60 | −0.041 |

**ALT queda 4.ª de 9** en la muestra completa; **3.ª en hombres y 5.ª en mujeres**, donde lidera la bilirrubina total. Las diferencias entre los cuatro primeros son pequeñas (0.394 a 0.371), así que tampoco hay una variable claramente dominante — pero ALT no encabeza en ningún corte.

### Confirmación independiente

Straw & Wu (2022), sobre **este mismo dataset** y con metodología distinta (importancia de variables en modelos entrenados), reportan que ALT y AST rankean **4.º–5.º en mujeres** y **7.º–8.º en hombres**, con albúmina y `A/G Ratio` por encima en el caso masculino. Y al rebalancear por sexo, **`Alkphos` y sexo pasan a ser las dos variables más importantes**.

Dos métodos independientes coinciden en que ALT no domina.

### ⚠️ Cómo NO leer esta tabla

**"Separa mejor" no equivale a "es clínicamente más importante".** `Selector` es un *proxy label*: el juicio de un especialista, no la verdad biológica.

Si quienes diagnosticaron se apoyaron sobre todo en la ictericia y la bilirrubina —el signo visible—, entonces `TB` y `DB` van a separar bien **por construcción**: no porque midan mejor el daño, sino porque **fueron el criterio con que se emitió la etiqueta**.

Esta tabla mide, en parte, *"en qué se fijó quien diagnosticó"*. Es el mismo problema de *label bias* ya registrado en la Fase 1, y aquí aparece en una forma nueva: **cualquier ranking de variables contra una etiqueta clínica hereda los criterios de quien la puso.**

### Qué queda establecido

1. El foco en ALT del proyecto **está justificado por la evidencia bibliográfica disponible, no por la fuerza de la señal**. Se declara explícitamente en vez de dejarlo implícito.
2. `Alkphos` es un **hilo a medio tirar**: 5.ª acá, pero primera para Straw & Wu tras rebalancear por sexo, y Loop C ya mostró que está confundida por edad. El obstáculo conocido es que la evidencia india sobre partición por sexo de ALP **se contradice entre estudios regionales**, así que probablemente no admita el mismo análisis de umbrales que ALT — y documentar esa imposibilidad ya es un resultado.
3. `TP` (proteína total) tiene una separación de **0.041**: prácticamente nula. Coincide con lo que T5 anticipó por otra vía —es la más redundante del eje de síntesis— y refuerza la recomendación de conservar `ALB` y `A/G Ratio` en su lugar.

## Valor agregado — ¿el cluster "severo" apunta a distinta etiología?

El Experimento 1 sí encontró un cluster de severidad multivariada
consistente y separable. Como valor agregado, se revisa si ese grupo
también se distingue en el cociente De Ritis (AST/ALT) — el mismo indicio
**sugerente, no probatorio** de patrón de daño que ya se exploró en Loop C
(`docs/fuentes/Consulta_3.md`).

In [8]:
de_ritis_rows = []

for stratum in STRATA:
    for variante in ("crudo", "log1p"):
        sub = exp1_results[(stratum, variante)]["sub"].copy()
        sub["DeRitis"] = de_ritis_ratio(sub)

        severe_id = sub.groupby("Cluster")["TB"].median().idxmax()
        severe = sub[sub["Cluster"] == severe_id]
        rest = sub[sub["Cluster"] != severe_id]

        de_ritis_rows.append({
            "Estrato": stratum,
            "variante": variante,
            # n SIEMPRE visible: una "mediana" de 1-4 personas no es una mediana
            "n_severo": len(severe),
            "n_resto": len(rest),
            "DR_med_severo": round(severe["DeRitis"].median(), 2),
            "DR_med_resto": round(rest["DeRitis"].median(), 2),
            "diferencia": round(severe["DeRitis"].median() - rest["DeRitis"].median(), 2),
            "n_suficiente": len(severe) >= MIN_CLUSTER_SIZE,
        })

de_ritis_summary = pd.DataFrame(de_ritis_rows).set_index(["Estrato", "variante"])
de_ritis_summary

n_severo  n_resto  DR_med_severo  DR_med_resto  \
Estrato        variante                                                   
18-39 · Female crudo            6       41           1.36          0.91   
               log1p           10       37           1.04          0.88   
18-39 · Male   crudo           19      131           1.28          1.20   
               log1p           29      121           1.68          1.11   
40-59 · Female crudo            4       61           1.22          1.19   
               log1p           23       42           1.36          1.03   
40-59 · Male   crudo           20      136           1.97          1.07   
               log1p           78       78           1.61          0.95   
60-120 · Male  crudo           13      100           1.56          1.27   
               log1p           22       91           1.70          1.22   

                         diferencia  n_suficiente  
Estrato        variante                            
18-39 · Female crudo           0.45          True  
               log1p           0.16          True  
18-39 · Male   crudo           0.08          True  
               log1p           0.57          True  
40-59 · Female crudo           0.03         False  
               log1p           0.33          True  
40-59 · Male   crudo           0.89          True  
               log1p           0.66          True  
60-120 · Male  crudo           0.29          True  
               log1p           0.48          True

**Interpretación — con `log1p` el patrón se vuelve consistente; con variables crudas era esporádico.**

### El resultado

| Estrato | Variante | n severo | De Ritis severo | De Ritis resto | Diferencia |
|---|---|---|---|---|---|
| 18-39 · Female | crudo | 6 | 1.36 | 0.91 | +0.45 |
| 18-39 · Female | **log1p** | 10 | 1.04 | 0.88 | +0.16 |
| 18-39 · Male | crudo | 19 | 1.28 | 1.20 | +0.08 |
| 18-39 · Male | **log1p** | 29 | 1.68 | 1.11 | **+0.57** |
| 40-59 · Female | crudo | 4 ⚠️ | 1.22 | 1.19 | +0.03 |
| 40-59 · Female | **log1p** | 23 | 1.36 | 1.03 | +0.33 |
| 40-59 · Male | crudo | 20 | 1.97 | 1.07 | **+0.89** |
| 40-59 · Male | **log1p** | 78 | 1.61 | 0.95 | **+0.66** |
| 60-120 · Male | crudo | 13 | 1.56 | 1.27 | +0.29 |
| 60-120 · Male | **log1p** | 22 | 1.70 | 1.22 | **+0.48** |

### Lo que cambió respecto de la versión anterior

En la primera versión de este notebook, la diferencia de De Ritis era **grande en solo 2 de 7 estratos** y se reportaba como "no sistemático". Pero aquellos 7 estratos incluían clusters de 1, 2 y 4 personas, cuyas "medianas" eran valores individuales.

**Con los estratos de muestra suficiente y la transformación adecuada, el patrón es consistente en los 5:** el cluster de mayor alteración tiene siempre un De Ritis más alto que el resto, con diferencias de **+0.16 a +0.66**.

Que un patrón aparezca en 5 de 5 en vez de en 2 de 7 es una diferencia sustantiva — y la razón por la que antes no se veía era metodológica, no biológica.

### Qué significa (con las mismas cautelas de Loop C)

Un De Ritis más alto en el grupo de mayor alteración es **compatible** con que ese grupo tenga más daño mitocondrial o fibrosis avanzada — los dos mecanismos que elevan AST relativo a ALT. Pero recordando lo que ya estableció `docs/fuentes/Consulta_3.md`:

- El cociente refleja **estadio** además de etiología. Un De Ritis más alto en el grupo más alterado puede significar simplemente **enfermedad más avanzada**, no una causa distinta.
- **Ningún estrato cruza el punto de corte clásico de 2.0** en la variante `log1p` (el máximo es 1.70). El valor de 1.97 en `40-59 · Male` crudo se queda justo debajo.
- Sin variable de etiología no hay forma de discriminar entre las lecturas.

**Formulación adoptada:** *"el grupo de mayor alteración bioquímica presenta consistentemente un cociente De Ritis más alto, lo cual es compatible tanto con mayor severidad como con distinta etiología — el dataset no permite distinguirlas."*

### Nota sobre la sensibilidad

`18-39 · Male` es el caso que más se mueve entre variantes: +0.08 crudo vs **+0.57** con `log1p`. Igual que en el Experimento 1, la conclusión depende de la transformación, y por eso se reportan ambas columnas en vez de elegir una.

⚠️ La fila marcada (`40-59 · Female`, crudo, n=4) es la que quedó por debajo de `MIN_CLUSTER_SIZE`; su diferencia de +0.03 **no es interpretable** y se muestra solo por completitud de la tabla de sensibilidad.

## Caveat de honestidad metodológica — qué prueba esto y qué no

| Pregunta | Respuesta |
|---|---|
| ¿El clustering encuentra estructura real? | **Sí.** En los 5 estratos analizados y con **ambas** transformaciones, el cluster de mayor `TB` tiene siempre más porcentaje de diagnóstico positivo que el resto. El algoritmo nunca ve `Selector` y aun así reagrupa de forma correlacionada con el juicio del médico. |
| ¿Los tamaños y las medianas de los clusters son estables? | **No.** Cambian sustancialmente entre variantes — `40-59 · Male` pasa de `[136, 20]` a `[78, 78]`. Por eso se reportan las dos y no se elige una. |
| ¿Se puede llamar "cluster" a cualquier grupo que devuelva el algoritmo? | **No.** Sin restricción de tamaño mínimo, el corte por mayor salto aísla *outliers* individuales. La versión anterior de este notebook reportaba "clusters" de 1 y 2 personas con sus "medianas". Corregido con `MIN_CLUSTER_SIZE = 5`. |
| ¿Produce puntos de corte diagnósticos utilizables? | **No.** Los rangos se superponen en 4 de 5 estratos; en el único que no, el corte queda **5.4× por encima** del umbral de literatura, por sesgo de muestreo (cohorte hospitalaria, no poblacional). |
| ¿Reemplaza los umbrales del ADR 0004? | **No, nunca.** Prati/ACG/AASLD siguen siendo la referencia del proyecto. Este notebook aporta la evidencia empírica de por qué esa decisión era la correcta. |
| ¿Qué pasa con `0-17` y `60-120 · Female`? | **Excluidos del clustering** (n=25 y n=20, bajo `MIN_STRATUM_SIZE_FOR_CLUSTERING`). No se les calcula ninguna estadística de cluster — no se reportan "con caveat", simplemente no se calculan. |
| ¿El análisis dice algo sobre mujeres mayores de 60? | **No, y eso es un resultado.** Solo hay 20 en el dataset. En un proyecto sobre equidad por sexo, el desbalance 441/142 **impide responder preguntas sobre las mujeres**, no solo sesga un modelo futuro. |
| ¿El De Ritis prueba distinta etiología? | **No.** El patrón es consistente en los 5 estratos con `log1p`, pero es igualmente compatible con "más severidad" que con "distinta causa" — y ningún estrato cruza el corte clásico de 2.0. Sin variable de etiología no se puede discriminar. |
| ¿Sirve para algo entonces? | Sí: (a) evidencia empírica que cierra la pregunta de los umbrales *data-driven*; (b) un *feature* candidato de severidad para la Fase 4+; (c) la demostración de que la elección de transformación **determina** qué encuentra el clustering en variables con asimetría de 10. |

## Notas para fases futuras (no se ejecuta nada aquí)

### Fase 3 — Preprocessing

- **La transformación logarítmica deja de ser opcional.** F3-R14 la proponía como alternativa [V] para las variables muy asimétricas. Este notebook muestra que **no es un adorno**: sin ella, un método basado en distancias euclidianas queda dominado por unos pocos pacientes extremos, hasta el punto de que en `40-59 · Female` **ningún corte del dendrograma** produce grupos en vez de individuos. Conviene reforzar ese requisito con la evidencia de acá.
- El tratamiento de outliers de T8 hereda el mismo problema: cualquier método que use distancias o varianzas sobre estas variables sin transformar está midiendo, sobre todo, a los casos extremos.

### Fase 4+ — Modelado (fuera de este PRD)

- El cluster de mayor alteración es candidato a *feature engineering* (p. ej. distancia al centroide del cluster severo del propio estrato edad-sexo), manteniendo la estratificación como referencia relativa en vez de comparar contra la población general.
- **Advertencia de fuga de información:** ese *feature* debe construirse **dentro** del `Pipeline`, ajustado solo con el *train set*. Calcular los centroides sobre el dataset completo filtraría información del test.
- Antes de usarlo, decidir **con qué transformación** se construye — este notebook muestra que la respuesta cambia el *feature*.

### Fase 5 — Auditoría de *fairness* (fuera de este PRD)

- **Limitación estructural que hay que declarar:** el análisis no puede decir nada sobre mujeres de 60+ (n=20) ni sobre menores (n=25). El desbalance de sexo no solo sesga modelos: **limita qué preguntas admiten respuesta**. Es un argumento adicional para la hipótesis de sub-diagnóstico de Loop A, y también un límite de la propia auditoría.
- Queda pendiente comparar la **composición por sexo** de los clusters de alteración: si mujeres con perfil bioquímico severo están sub-representadas en `Selector` positivo respecto de hombres con perfil comparable, sería evidencia adicional (no prueba) de sub-diagnóstico.
- **Hilo abierto sobre ALP.** Straw & Wu encuentran que `Alkphos` y sexo pasan a ser las dos variables más importantes al rebalancear por sexo; Loop C encontró que ALP está confundida por edad. Son el mismo hilo y está a medio tirar. El obstáculo conocido: la evidencia india sobre si ALP requiere partición por sexo **se contradice entre estudios regionales** (ver `Consulta_1.md`), así que probablemente **no** se le pueda aplicar el mismo análisis de sensibilidad de umbrales que a ALT. Documentar esa imposibilidad ya es un resultado.

### Sobre derivar umbrales de datos

Si en algún momento se consigue una muestra con una franja amplia de personas sanas, repetir este clustering ahí sería la forma correcta de intentar cortes *data-driven*. **Con esta cohorte hospitalaria no es posible**, y este notebook lo demuestra en vez de suponerlo.